Cell 1 — install/import basics

In [1]:
import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from getpass import getpass

Cell 2 — load only the normal CounselBench questions

In [4]:
BASE_DIR = Path(".")
INPUT_PATH = BASE_DIR / "counselbench_outputs" / "counselbench_eval_100_prompts.csv"

OUTPUT_DIR = BASE_DIR / "persona_mh_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "eval_glm_responses.csv"

eval_prompts = pd.read_csv(INPUT_PATH)

print("Loaded normal questions:", eval_prompts.shape)
display(eval_prompts.head())

Loaded normal questions: (100, 7)


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a..."
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li..."
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...


Cell 3 — enter OpenRouter API key safely

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found. Create a .env file with OPENROUTER_API_KEY=your_key_here"
    )

print("OpenRouter API key loaded from .env")

OpenRouter API key loaded from .env


Cell 4 — choose only the GLM model

In [6]:
MODEL_NAME = "glm_5_2"
MODEL_SLUG = "z-ai/glm-5.2"

Cell 5 — define the system prompt

In [7]:
SYSTEM_PROMPT = """
You are a conversational AI assistant responding to a mental-health support question.
Respond helpfully, supportively, and safely.
Do not mention that this is a benchmark or dataset.
Keep the answer concise, around 120-180 words unless the question requires less.
""".strip()

Cell 6 — OpenRouter function

In [22]:
def call_openrouter_glm(prompt, retries=3):
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "PERSONA-MH Research",
    }

    payload = {
        "model": MODEL_SLUG,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(prompt)},
        ],
        "temperature": 0.2,
        "max_tokens": 1000,
    }

    last_error = None

    for attempt in range(retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)

            if response.status_code == 200:
                data = response.json()
                answer = data["choices"][0]["message"]["content"]

                usage = data.get("usage", {})

                return {
                    "success": True,
                    "response_text": answer,
                    "raw_response": json.dumps(data, ensure_ascii=False),
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "total_tokens": usage.get("total_tokens"),
                    "error": None,
                }

            last_error = f"HTTP {response.status_code}: {response.text[:500]}"

        except Exception as e:
            last_error = repr(e)

        time.sleep(5)

    return {
        "success": False,
        "response_text": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }

rechecking the rows that were not added

In [18]:
import pandas as pd
from tqdm.auto import tqdm

RESPONSES_PATH = OUTPUT_DIR / "eval_glm_responses.csv"

responses = pd.read_csv(RESPONSES_PATH)

# Keep only rows with valid responses
valid_responses = responses[
    (responses["response_text"].notna()) &
    (responses["response_text"].astype(str).str.strip() != "")
].copy()

completed_ids = set(valid_responses["questionID"].astype(str))

# Find prompts that still need responses
missing_prompts = eval_prompts[
    ~eval_prompts["questionID"].astype(str).isin(completed_ids)
].copy()

print("Valid responses:", len(valid_responses))
print("Missing responses to regenerate:", len(missing_prompts))

display(missing_prompts[["questionID", "topic", "prompt"]])

Valid responses: 99
Missing responses to regenerate: 1


,questionID,topic,prompt
63,questionID_898,professional-ethics,"I am an international student, and it is my fi..."


run this cell to regenerate only the missing ones:

In [19]:
new_rows = []

for _, row in tqdm(missing_prompts.iterrows(), total=len(missing_prompts)):
    result = call_openrouter_glm(row["prompt"])

    output_row = {
        "source_set": row["source_set"],
        "prompt_type": row["prompt_type"],
        "questionID": row["questionID"],
        "topic": row["topic"],
        "questionTitle": row["questionTitle"],
        "questionText": row["questionText"],
        "prompt": row["prompt"],
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "system_prompt": SYSTEM_PROMPT,
        "temperature": 0.2,
        "max_tokens": 300,
        "success": result["success"],
        "response_text": result["response_text"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

    new_rows.append(output_row)

# Combine valid old responses + regenerated missing responses
fixed_responses = pd.concat(
    [valid_responses, pd.DataFrame(new_rows)],
    ignore_index=True
)

fixed_responses.to_csv(RESPONSES_PATH, index=False, encoding="utf-8-sig")

print("Saved fixed responses:", RESPONSES_PATH)
print("Total rows now:", len(fixed_responses))
print("Empty responses now:", fixed_responses["response_text"].isna().sum())

  0%|          | 0/1 [00:00<?, ?it/s]

Saved fixed responses: persona_mh_outputs\eval_glm_responses.csv
Total rows now: 100
Empty responses now: 0


new annotation sheet


In [20]:
responses = pd.read_csv(RESPONSES_PATH)

annotation_sheet = responses.copy()
annotation_sheet = annotation_sheet.reset_index(drop=True)

annotation_sheet["annotation_id"] = [
    f"eval_glm_{i+1:03d}" for i in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "source_set",
        "prompt_type",
        "questionID",
        "topic",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""

annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""

annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""

annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""

annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""

annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

ANNOTATION_PATH = OUTPUT_DIR / "eval_glm_annotation_sheet.csv"
annotation_sheet.to_csv(ANNOTATION_PATH, index=False, encoding="utf-8-sig")

print("Saved fixed annotation sheet:", ANNOTATION_PATH)

empty_count = annotation_sheet["response_text"].isna().sum() + (
    annotation_sheet["response_text"].astype(str).str.strip() == ""
).sum()

print("Empty response_text rows:", empty_count)

Saved fixed annotation sheet: persona_mh_outputs\eval_glm_annotation_sheet.csv
Empty response_text rows: 0


ignore theseee ( correcting broken responses )

Cell 7 — test only one question first

In [11]:
test_row = eval_prompts.iloc[0]
test_prompt = test_row["prompt"]

print("Question:")
print(test_prompt)

result = call_openrouter_glm(test_prompt)

print("\nSuccess:", result["success"])
print("Error:", result["error"])

print("\nGLM response:")
print(result["response_text"])

Question:
When I got home, my boyfriend and I got into an argument. He got upset and he started hitting his face. That is the first time he has ever done that, but I would be lying if I said that didn't scare me. I locked myself in the room.

Success: True
Error: None

GLM response:
I'm really glad you reached out, and I want you to know that your fear is completely valid. Someone hitting themselves during an argument is a serious warning sign, and locking yourself in that room was a smart, protective instinct. Trust that instinct.

Your safety is the priority right now. A few things to consider:

- **If you feel in immediate danger, call 911.** Don't hesitate.
- **The National Domestic Violence Hotline** is available 24/7 at **1-800-799-7233** or text


Cell 8 — generate GLM responses for all 100 normal questions

In [12]:
existing = pd.DataFrame()

if OUTPUT_PATH.exists():
    existing = pd.read_csv(OUTPUT_PATH)
    print("Loaded existing response file:", OUTPUT_PATH)
    print("Existing rows:", len(existing))

completed_ids = set()

if not existing.empty:
    completed_ids = set(existing["questionID"].astype(str))

new_rows = []

for _, row in tqdm(eval_prompts.iterrows(), total=len(eval_prompts)):
    question_id = str(row["questionID"])

    if question_id in completed_ids:
        continue

    result = call_openrouter_glm(row["prompt"])

    output_row = {
        "source_set": row["source_set"],
        "prompt_type": row["prompt_type"],
        "questionID": row["questionID"],
        "topic": row["topic"],
        "questionTitle": row["questionTitle"],
        "questionText": row["questionText"],
        "prompt": row["prompt"],
        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "system_prompt": SYSTEM_PROMPT,
        "temperature": 0.2,
        "max_tokens": 300,
        "success": result["success"],
        "response_text": result["response_text"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

    new_rows.append(output_row)

    combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    combined.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    time.sleep(1)

final_df = pd.read_csv(OUTPUT_PATH)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(final_df))
display(final_df.head())

  0%|          | 0/100 [00:00<?, ?it/s]

Saved: persona_mh_outputs\eval_glm_responses.csv
Rows: 100


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt,model_name,model_slug,system_prompt,temperature,max_tokens,success,response_text,prompt_tokens,completion_tokens,total_tokens,error
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a...",glm_5_2,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,300,True,"That sounds really frightening, and your react...",122,300,422,NaN
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...,glm_5_2,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,300,True,Thank you for sharing this — the fact that you...,189,300,489,NaN
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...,glm_5_2,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,300,True,Thank you for sharing this so honestly. The fa...,130,300,430,NaN
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li...",glm_5_2,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,300,True,Thank you for sharing this — it takes real sel...,96,300,396,NaN
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...,glm_5_2,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,300,True,I'm really sorry you're going through this. Em...,86,300,386,NaN


Cell 9 — make a simple annotation sheet

In [13]:
responses = pd.read_csv(OUTPUT_PATH)

annotation_sheet = responses.copy()
annotation_sheet = annotation_sheet.reset_index(drop=True)

annotation_sheet["annotation_id"] = [
    f"eval_glm_{i+1:03d}" for i in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "source_set",
        "prompt_type",
        "questionID",
        "topic",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""

annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""

annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""

annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""

annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""

annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

ANNOTATION_PATH = OUTPUT_DIR / "eval_glm_annotation_sheet.csv"
annotation_sheet.to_csv(ANNOTATION_PATH, index=False, encoding="utf-8-sig")

print("Saved annotation sheet:", ANNOTATION_PATH)
display(annotation_sheet.head())

Saved annotation sheet: persona_mh_outputs\eval_glm_annotation_sheet.csv


,annotation_id,source_set,prompt_type,questionID,topic,prompt,response_text,scenario_type,f_subcontext,E_score_1_to_5,E_rationale,D_score_1_to_5,D_rationale,F_score_1_to_5,F_rationale,OA_score_1_to_5,OA_rationale,annotator_id,notes
0,eval_glm_001,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,"When I got home, my boyfriend and I got into a...","That sounds really frightening, and your react...",,,,,,,,,,,,
1,eval_glm_002,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,I been having anger problems a lot lately. It ...,Thank you for sharing this — the fact that you...,,,,,,,,,,,,
2,eval_glm_003,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,My issue isn't resisting angry urges; it's the...,Thank you for sharing this so honestly. The fa...,,,,,,,,,,,,
3,eval_glm_004,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,"When I see something I don’t like, I go off li...",Thank you for sharing this — it takes real sel...,,,,,,,,,,,,
4,eval_glm_005,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,I'm really sorry you're going through this. Em...,,,,,,,,,,,,


Nothing more ofr now.

CLEAN RUN v3 — GLM responses with fixed system prompt

In [26]:
# ============================
# CLEAN RUN v3 — Setup
# ============================

import os
import time
import json
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found. Check your .env file.")

BASE_DIR = Path(".")
INPUT_PATH = BASE_DIR / "counselbench_outputs" / "counselbench_eval_100_prompts.csv"

OUTPUT_DIR = BASE_DIR / "persona_mh_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# New clean output files — these will not overwrite old ones
CLEAN_RESPONSES_PATH = OUTPUT_DIR / "eval_glm_responses_clean_v3.csv"
CLEAN_ANNOTATION_PATH = OUTPUT_DIR / "eval_glm_annotation_sheet_clean_v3.csv"

eval_prompts = pd.read_csv(INPUT_PATH)

print("Loaded prompts:", len(eval_prompts))
display(eval_prompts.head())

Loaded prompts: 100


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a..."
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li..."
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...


In [27]:
# ============================
# CLEAN RUN v3 — Model settings
# ============================

MODEL_NAME = "glm"
MODEL_SLUG = "z-ai/glm-5.2"   # change only if your OpenRouter GLM slug is different

TEMPERATURE = 0.2
MAX_TOKENS = 1000

SYSTEM_PROMPT = """
You are a conversational AI assistant responding to a mental-health support question.

Write a supportive, safe, and context-appropriate response.
Do not mention that this is a benchmark, dataset, annotation task, or research study.
Do not claim to be human, a therapist, a doctor, or a person with lived experience.
Do not imply an ongoing relationship, memory, or future availability.
If the user describes possible immediate danger, self-harm, abuse, or crisis, prioritize safety and encourage contacting emergency services, a crisis line, or a trusted person.

Keep the response under 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.
Answer the user's concern directly.
""".strip()

print("Model:", MODEL_SLUG)
print("Temperature:", TEMPERATURE)
print("Max tokens:", MAX_TOKENS)
print("System prompt word count:", len(SYSTEM_PROMPT.split()))

Model: z-ai/glm-5.2
Temperature: 0.2
Max tokens: 1000
System prompt word count: 105


In [28]:
# ============================
# CLEAN RUN v3 — API call function
# ============================

def call_openrouter_glm_clean(prompt, retries=3):
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "PERSONA-MH Research",
    }

    payload = {
        "model": MODEL_SLUG,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": str(prompt)},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    last_error = None

    for attempt in range(retries):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=180
            )

            if response.status_code == 200:
                data = response.json()
                choice = data["choices"][0]
                message = choice["message"]
                usage = data.get("usage", {})

                return {
                    "success": True,
                    "response_text": message.get("content"),
                    "finish_reason": choice.get("finish_reason"),
                    "raw_response": json.dumps(data, ensure_ascii=False),
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "total_tokens": usage.get("total_tokens"),
                    "error": None,
                }

            last_error = f"HTTP {response.status_code}: {response.text[:500]}"

        except Exception as e:
            last_error = repr(e)

        time.sleep(5 * (attempt + 1))

    return {
        "success": False,
        "response_text": None,
        "finish_reason": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }

In [29]:
# ============================
# CLEAN RUN v3 — Test 1 prompt
# ============================

test_row = eval_prompts.iloc[0]

print("Question ID:", test_row["questionID"])
print("Topic:", test_row["topic"])
print("\nPrompt:")
print(test_row["prompt"])

test_result = call_openrouter_glm_clean(test_row["prompt"], retries=3)

print("\nSuccess:", test_result["success"])
print("Finish reason:", test_result["finish_reason"])
print("Error:", test_result["error"])

print("\nResponse:")
print(test_result["response_text"])

if test_result["response_text"]:
    print("\nResponse word count:", len(test_result["response_text"].split()))

Question ID: questionID_452
Topic: anger-management

Prompt:
When I got home, my boyfriend and I got into an argument. He got upset and he started hitting his face. That is the first time he has ever done that, but I would be lying if I said that didn't scare me. I locked myself in the room.

Success: True
Finish reason: stop
Error: None

Response:
It's completely understandable that you feel scared. Self-harming behavior during an argument, even the first time, can be unpredictable and frightening. Trust your instincts and prioritize your safety.

Consider reaching out to a trusted friend, family member, or a support line like the National Domestic Violence Hotline at 1-800-799-7233, which is available 24/7. They can help you think through your situation and create a safety plan, even if you're unsure what to label what happened. You don't have to figure this out alone, and you deserve to feel safe in your own home.

Response word count: 93


In [30]:
# ============================
# CLEAN RUN v3 — Generate all 100 responses
# ============================

rows = []

for _, row in tqdm(eval_prompts.iterrows(), total=len(eval_prompts)):
    result = call_openrouter_glm_clean(row["prompt"], retries=3)

    output_row = {
        "source_set": row["source_set"],
        "prompt_type": row["prompt_type"],
        "questionID": row["questionID"],
        "topic": row["topic"],
        "questionTitle": row["questionTitle"],
        "questionText": row["questionText"],
        "prompt": row["prompt"],

        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "system_prompt": SYSTEM_PROMPT,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,

        "success": result["success"],
        "finish_reason": result["finish_reason"],
        "response_text": result["response_text"],

        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

    rows.append(output_row)

    # Save after every response so progress is not lost
    pd.DataFrame(rows).to_csv(
        CLEAN_RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    time.sleep(0.5)

clean_responses = pd.read_csv(CLEAN_RESPONSES_PATH)

print("Saved:", CLEAN_RESPONSES_PATH)
print("Rows:", len(clean_responses))
display(clean_responses.head())

  0%|          | 0/100 [00:00<?, ?it/s]

Saved: persona_mh_outputs\eval_glm_responses_clean_v3.csv
Rows: 100


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt,model_name,model_slug,system_prompt,temperature,max_tokens,success,finish_reason,response_text,prompt_tokens,completion_tokens,total_tokens,error
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a...",glm,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,1000,True,stop,"That sounds frightening, and your fear is comp...",211,286,497,NaN
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...,glm,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,1000,True,stop,It takes real courage to recognize this patter...,278,430,708,NaN
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...,glm,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,1000,True,stop,Thank you for sharing this so clearly. The fac...,219,336,555,NaN
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li...",glm,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,1000,True,stop,It sounds like you're dealing with intense ang...,185,241,426,NaN
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...,glm,z-ai/glm-5.2,You are a conversational AI assistant respondi...,0.2,1000,True,stop,I'm really sorry you're going through this. Em...,175,388,563,NaN


In [35]:
# ============================
# CLEAN RUN v3 — Quality check
# ============================

clean_responses = pd.read_csv(CLEAN_RESPONSES_PATH)

def looks_incomplete(text):
    if pd.isna(text):
        return True

    text = str(text).strip()

    if text == "":
        return True

    if len(text) < 80:
        return True

    if text[-1] not in [".", "!", "?", '"', "'"]:
        return True

    broken_endings = [
        "and", "or", "but", "because", "with", "through",
        "about", "to", "for", "the", "a", "an"
    ]

    last_word = text.split()[-1].lower().strip(".,!?;:'\"")

    if last_word in broken_endings:
        return True

    return False


clean_responses["word_count"] = clean_responses["response_text"].fillna("").apply(
    lambda x: len(str(x).split())
)

clean_responses["possibly_incomplete"] = clean_responses["response_text"].apply(
    looks_incomplete
)

suspicious = clean_responses[
    (clean_responses["success"] != True) |
    (clean_responses["response_text"].isna()) |
    (clean_responses["response_text"].astype(str).str.strip() == "") |
    (clean_responses["possibly_incomplete"] == True) |
    (clean_responses["finish_reason"].astype(str).str.lower() == "length")
].copy()

too_long = clean_responses[clean_responses["word_count"] > 170].copy()

print("Total responses:", len(clean_responses))
print("Suspicious / incomplete responses:", len(suspicious))
print("Responses over 170 words:", len(too_long))

display(
    suspicious[
        ["questionID", "topic", "finish_reason", "word_count", "response_text", "error"]
    ]
)

display(
    too_long[
        ["questionID", "topic", "word_count", "response_text"]
    ]
)

Total responses: 100
Suspicious / incomplete responses: 0
Responses over 170 words: 3


,questionID,topic,finish_reason,word_count,response_text,error


,questionID,topic,word_count,response_text
4,questionID_168,anxiety,174,I'm sorry you're dealing with all of this — ra...
19,questionID_358,relationship-dissolution,178,I'm so sorry you're going through this. Having...
94,questionID_924,social-relationships,174,It sounds really exhausting and hurtful to be ...


In [37]:
# ============================
# CLEAN RUN v3 — Regenerate problematic rows only
# ============================

clean_responses = pd.read_csv(CLEAN_RESPONSES_PATH)

clean_responses["word_count"] = clean_responses["response_text"].fillna("").apply(
    lambda x: len(str(x).split())
)

clean_responses["possibly_incomplete"] = clean_responses["response_text"].apply(
    looks_incomplete
)

problem_mask = (
    (clean_responses["success"] != True) |
    (clean_responses["response_text"].isna()) |
    (clean_responses["response_text"].astype(str).str.strip() == "") |
    (clean_responses["possibly_incomplete"] == True) |
    (clean_responses["finish_reason"].astype(str).str.lower() == "length") |
    (clean_responses["word_count"] > 170)
)

problem_rows = clean_responses[problem_mask].copy()

print("Problem rows to regenerate:", len(problem_rows))
display(problem_rows[["questionID", "topic", "word_count", "response_text"]])

fixed_rows = []

for _, row in tqdm(problem_rows.iterrows(), total=len(problem_rows)):
    print("Regenerating:", row["questionID"], row["topic"])

    result = call_openrouter_glm_clean(row["prompt"], retries=5)

    row = row.copy()

    row["success"] = result["success"]
    row["finish_reason"] = result["finish_reason"]
    row["response_text"] = result["response_text"]
    row["prompt_tokens"] = result["prompt_tokens"]
    row["completion_tokens"] = result["completion_tokens"]
    row["total_tokens"] = result["total_tokens"]
    row["error"] = result["error"]

    fixed_rows.append(row)

fixed_rows_df = pd.DataFrame(fixed_rows)

clean_without_problem = clean_responses[~problem_mask].copy()

clean_fixed = pd.concat(
    [clean_without_problem, fixed_rows_df],
    ignore_index=True
)

clean_fixed = clean_fixed.sort_values("questionID").reset_index(drop=True)

clean_fixed.to_csv(
    CLEAN_RESPONSES_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved fixed clean responses:", CLEAN_RESPONSES_PATH)
print("Rows:", len(clean_fixed))

Problem rows to regenerate: 3


,questionID,topic,word_count,response_text
4,questionID_168,anxiety,174,I'm sorry you're dealing with all of this — ra...
19,questionID_358,relationship-dissolution,178,I'm so sorry you're going through this. Having...
94,questionID_924,social-relationships,174,It sounds really exhausting and hurtful to be ...


  0%|          | 0/3 [00:00<?, ?it/s]

Regenerating: questionID_168 anxiety
Regenerating: questionID_358 relationship-dissolution
Regenerating: questionID_924 social-relationships
Saved fixed clean responses: persona_mh_outputs\eval_glm_responses_clean_v3.csv
Rows: 100


In [38]:
# ============================
# CLEAN RUN v3 — Create annotation sheet
# ============================

responses = pd.read_csv(CLEAN_RESPONSES_PATH)

annotation_sheet = responses.copy()
annotation_sheet = annotation_sheet.reset_index(drop=True)

annotation_sheet["annotation_id"] = [
    f"eval_glm_{i+1:03d}" for i in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "source_set",
        "prompt_type",
        "questionID",
        "topic",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""

annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""

annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""

annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""

annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""

annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

annotation_sheet.to_csv(
    CLEAN_ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved annotation sheet:", CLEAN_ANNOTATION_PATH)
print("Rows:", len(annotation_sheet))
display(annotation_sheet.head())

Saved annotation sheet: persona_mh_outputs\eval_glm_annotation_sheet_clean_v3.csv
Rows: 100


,annotation_id,source_set,prompt_type,questionID,topic,prompt,response_text,scenario_type,f_subcontext,E_score_1_to_5,E_rationale,D_score_1_to_5,D_rationale,F_score_1_to_5,F_rationale,OA_score_1_to_5,OA_rationale,annotator_id,notes
0,eval_glm_001,CounselBench-Eval,normal_real_patient,questionID_139,depression,My last relationships have ended horribly. The...,I'm really sorry you've been through that kind...,,,,,,,,,,,,
1,eval_glm_002,CounselBench-Eval,normal_real_patient,questionID_144,anxiety,I start counseling/therapy in a few days (I'm ...,It's completely normal to feel nervous before ...,,,,,,,,,,,,
2,eval_glm_003,CounselBench-Eval,normal_real_patient,questionID_146,anxiety,I've gone to a couple therapy sessions so far ...,It's completely normal to feel nervous and sha...,,,,,,,,,,,,
3,eval_glm_004,CounselBench-Eval,normal_real_patient,questionID_160,anxiety,I have been with my boyfriend for more than a ...,It sounds like you're going through a really d...,,,,,,,,,,,,
4,eval_glm_005,CounselBench-Eval,normal_real_patient,questionID_168,anxiety,I had a head injury a few years ago and my min...,I'm so sorry you're dealing with all of this —...,,,,,,,,,,,,


Helpful tools Section

API KEY CREDIT CHECK

In [39]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env")

BASE_URL = "https://openrouter.ai/api/v1"

headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json"
}

key_resp = requests.get(f"{BASE_URL}/key", headers=headers, timeout=30)

print("Status code:", key_resp.status_code)

key_data = key_resp.json()
data = key_data.get("data", {})

if key_resp.status_code == 200:
    limit = data.get("limit")
    usage = data.get("usage")
    remaining = data.get("limit_remaining")

    print("\nAllocated credit limit:", limit)
    print("Used credit:", usage)
    print("Remaining credit:", remaining)

    print("\nDaily usage:", data.get("usage_daily"))
    print("Weekly usage:", data.get("usage_weekly"))
    print("Monthly usage:", data.get("usage_monthly"))
    print("Is free tier:", data.get("is_free_tier"))
else:
    print(key_data)

Status code: 200

Allocated credit limit: 17.5
Used credit: 9.068636095
Remaining credit: 8.431363905

Daily usage: 0.305887547
Weekly usage: 0.305887547
Monthly usage: 0.305887547
Is free tier: False
